# 04 — Phase 3: GQA + KIVI (2-bit KV Cache)

**技术**: KIVI — 2-bit 非对称量化 KV Cache  
**框架**: PyTorch 原生 + KIVI CUDA Backend（`kivi_gemv`）

> ⚠️ 本 notebook 使用 **PyTorch 原生推理路径**（`metrics.py`），不经过 vLLM。
> KIVI 和 vLLM 的 PagedAttention 在同一层（KV Cache 管理层）做完全不同的事情，
> 两者无法并行运行。这是控制变量法的第三组：只改 **数值精度** 这一个变量。

### KIVI 原理
- **K Cache**: per-channel 量化（沿 head_dim 方向统计 min/max）
- **V Cache**: per-token 量化（沿 seq_len 方向统计 min/max）
- **Residual window**: 最近 128 个 token 保持 FP16（保证质量）
- 理论压缩 **~5-8×**

### CUDA Backend
KIVI 的自定义 CUDA kernel（`quant/csrc/gemv_cuda.cu`）提供：
- **`bgemv2_kernel_outer_dim`**: 2-bit packed GEMV（Q × K_quant, attn × V_quant）
- 每个 warp 读 32 个 packed uint32，在线反量化 `dequant = scale * (int & 0x3) + zero`
- 支持 GQA：通过 `nh / nh_kv` ratio 自动处理 KV head 复用

编译方法（在 l4t-pytorch 容器内）：
```bash
cd ~/KIVI && pip install -e .
cd quant && TORCH_CUDA_ARCH_LIST="8.7" pip install -e .
python -c "import kivi_gemv; print('OK')"
```

### 预期效果
- KV Cache 显存占用从 FP16 降低 5-8×
- **TPOT 降低**: decode 每步搬运的 KV 数据量减少（显存带宽是 Jetson 瓶颈）
- **TTFT 基本不变**: prefill 阶段 KV 尚未量化
- **PPL 可能略升**: 2-bit 有损压缩 → 需要检查质量退化

In [1]:
import sys, gc, torch
sys.path.insert(0, '..')

from src.metrics import (
    measure_generation, run_benchmark, find_oom_threshold,
    print_memory_budget, JETSON_USABLE_GB,
)
from src.kivi_wrapper import (
    create_kivi_cache, is_cuda_backend_available, get_backend_info,
)
from src.perplexity import compute_ppl_with_kv_cache
from src.dataset_utils import load_prompts

print(f"KIVI Backend: {get_backend_info()}")
if not is_cuda_backend_available():
    print("\n⚠️  KIVI CUDA 后端未编译。使用 Python 参考实现。")
    print("    真正的延迟收益需要 CUDA kernel!")
    print("    编译方法: 见 scripts/setup_kivi_jetson.sh")

kivi_gemv is available, but no KIVI cache class was found in Python package. Will use local KIVICache fallback for cache object path.


KIVI Backend: KIVI CUDA backend (kivi_gemv only; cache class missing)


In [2]:

from src.jetson_utils import load_model_safe, aggressive_cleanup

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
FALLBACK_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
DEVICE = "cuda"

model, tokenizer = load_model_safe(
    MODEL_NAME,
    fallback_name=FALLBACK_NAME,
    device=DEVICE,
)
budget = print_memory_budget(model, DEVICE)


/opt/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/venv/lib/python3.12/site-packages/transformers/utils/hub.py:105: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


✓ Loaded Qwen/Qwen2.5-1.5B-Instruct (torch.float16)  GPU mem: 2.88 GB
Jetson Orin NX Memory Budget
Total physical    : 16,384 MB (16.0 GB)
Usable (after OS) : 11,264 MB (11.0 GB)
Model weights     : 2,944 MB
CUDA overhead     : ~500 MB
>>> KV Cache budget: 7,820 MB (7.6 GB)
Currently allocated: 2,944 MB
Currently reserved : 3,056 MB


In [3]:
# 替换 Qwen2 Attention 为 KIVI 路径
from src.qwen_kivi_2 import patch_qwen_with_kivi_2
model = patch_qwen_with_kivi_2(model)

AttributeError: 'Qwen2Attention' object has no attribute 'num_heads'

In [ ]:
# KIVI 参数
BITS = 2
GROUP_SIZE = 32
RESIDUAL_LENGTH = 128

prompts = load_prompts('../results/ehr_prompts.json')
print(f"Loaded {len(prompts)} prompts")
print(f"\nKIVI config: bits={BITS}, group_size={GROUP_SIZE}, residual={RESIDUAL_LENGTH}")

In [ ]:
import kivi_gemv
print("KIVI C++ 引擎内部接口:")
for func in dir(kivi_gemv):
    if not func.startswith('__'):
        print(f" - {func}")

---
## OOM 阈值（KIVI 应该能撑更长）

In [ ]:
print("KIVI OOM 探测...")
oom_kivi = find_oom_threshold(
    model, tokenizer,
    context_lengths=[256, 512, 1024, 2048, 4096],
    max_new_tokens=16,
    cache_factory=lambda: create_kivi_cache(
        residual_length=RESIDUAL_LENGTH,
        group_size=GROUP_SIZE,
        bits=BITS,
    ),
    memory_headroom_mb=1500,
)

print(f"\n最大安全长度: {oom_kivi['max_safe_length']}")
print(f"OOM 发生在  : {oom_kivi['oom_length']}")
for r in oom_kivi['results']:
    if r['status'] == 'ok':
        print(f"  ctx={r['context_length']:>6}  peak={r['peak_memory_mb']:>7.0f} MB  "
              f"util={r['utilization']*100:>5.1f}%")
    else:
        print(f"  ctx={r['context_length']:>6}  → {r['status']}")

In [ ]:
import gc
import torch
import sys

def force_clear_memory():
    print("🧹 正在启动深度内存清理序列...")
    
    # 步骤 1: 扫描并删除当前全局命名空间中占用内存的大块头变量
    # (你可以根据你实际的代码，在这里添加模型或 Cache 的变量名)
    target_vars = ['model', 'kivi_cache', 'outputs', 'input_ids', 'past_key_values', 'dummy_v']
    deleted_count = 0
    for var_name in target_vars:
        if var_name in globals():
            del globals()[var_name]
            deleted_count += 1
            
    print(f"🚮 删除了 {deleted_count} 个大型全局变量引用。")

    # 步骤 2: 强制触发 Python 底层的垃圾回收机制 (非常关键，必须在 empty_cache 之前)
    gc.collect()
    
    # 步骤 3: 释放 PyTorch 显存分配器 (Allocator) 缓存的死区内存
    if torch.cuda.is_available():
        # 清空常规显存缓存
        torch.cuda.empty_cache()
        # 清空进程间通信可能残留的显存 (针对 Jetson 统一内存架构特别有效)
        torch.cuda.ipc_collect() 
        
        # 步骤 4: 打印清理后的真实显存状态
        allocated = torch.cuda.memory_allocated() / (1024 ** 3)
        reserved = torch.cuda.memory_reserved() / (1024 ** 3)
        print(f"✅ 显存清理完毕!")
        print(f"📊 当前正在使用 (Allocated): {allocated:.3f} GB")
        print(f"📊 显存池总保留 (Reserved):  {reserved:.3f} GB")
    else:
        print("✅ 内存清理完毕 (未检测到 CUDA 环境)。")

# 立即执行
force_clear_memory()

In [ ]:
# 单样本快速测试 (先 warmup)
print("Warmup...")
_ = measure_generation(model, tokenizer, prompts[0]['prompt'],
                       max_new_tokens=8,
                       cache_impl=create_kivi_cache(RESIDUAL_LENGTH, GROUP_SIZE, BITS))
torch.cuda.empty_cache()

cache = create_kivi_cache(RESIDUAL_LENGTH, GROUP_SIZE, BITS)
m = measure_generation(model, tokenizer, prompts[0]['prompt'],
                       max_new_tokens=64, cache_impl=cache)

print(f"TTFT           : {m.ttft_ms:.1f} ms")
print(f"TPOT           : {m.tpot_ms:.1f} ms")
print(f"KV Cache (est) : {m.kv_cache_memory_mb:.1f} MB")
print(f"Peak memory    : {m.peak_memory_mb:.0f} MB")
print(f"Memory util    : {m.memory_utilization*100:.1f}%")
print(f"\nCache info: {cache}")

In [ ]:
# 完整 benchmark
results_kivi = run_benchmark(
    model, tokenizer, prompts,
    cache_factory=lambda: create_kivi_cache(RESIDUAL_LENGTH, GROUP_SIZE, BITS),
    max_new_tokens=256,
    warmup_runs=2,
)
print(f"\nCompleted {len(results_kivi)} samples")

In [ ]:
import pandas as pd

df = pd.DataFrame(results_kivi)
df['config'] = 'GQA+KIVI'
df.to_csv('../results/gqa_kivi.csv', index=False)

print(df[['ttft_ms', 'tpot_ms', 'peak_memory_mb',
           'kv_cache_memory_mb', 'memory_utilization']].describe().round(1))

---
## PPL 退化评估

2-bit 量化可能导致生成质量下降。PPL 增加 > 5% 视为显著退化。

In [ ]:
# 用 KIVI cache 跑 PPL
eval_texts = [p['reference_answer'] for p in prompts[:20] if p.get('reference_answer')]

ppl_kivi = compute_ppl_with_kv_cache(
    model, tokenizer, eval_texts,
    cache_factory=lambda: create_kivi_cache(RESIDUAL_LENGTH, GROUP_SIZE, BITS),
    max_length=512,
)

# 加载 baseline PPL
import json
with open('../results/ppl_results.json') as f:
    ppl_all = json.load(f)

baseline_ppl = ppl_all['GQA_only']['ppl']
kivi_ppl = ppl_kivi['ppl']
degradation = (kivi_ppl - baseline_ppl) / baseline_ppl * 100

print(f"Baseline PPL : {baseline_ppl:.2f}")
print(f"KIVI 2-bit PPL: {kivi_ppl:.2f}")
print(f"退化         : {degradation:+.1f}%")
if abs(degradation) < 5:
    print("✓ 2-bit 量化质量可接受")
else:
    print("⚠️  PPL 退化超过 5%，可能影响医疗推理质量")

# 保存 JSON
ppl_all['GQA+KIVI'] = ppl_kivi
with open('../results/ppl_results.json', 'w') as f:
    json.dump(ppl_all, f, indent=2)

# 保存 CSV
ppl_df = pd.DataFrame([{
    'config': 'GQA+KIVI',
    'ppl': ppl_kivi['ppl'],
    'avg_loss': ppl_kivi['avg_loss'],
    'num_tokens': ppl_kivi['num_tokens'],
}])
ppl_df.to_csv('../results/ppl_gqa_kivi.csv', index=False)
print("Saved → results/ppl_results.json + results/ppl_gqa_kivi.csv")

In [ ]:
# 生成质量抽检
print("=== KIVI 2-bit 生成示例 ===")
for i in [0, 10, 20]:
    if i < len(results_kivi):
        print(f"\n--- Sample {i}: {results_kivi[i].get('question', '')[:60]}... ---")
        print(results_kivi[i].get('sample_text', '')[:300])

In [ ]:
del model
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory after cleanup: {torch.cuda.memory_allocated() / (1024**2):.0f} MB")